In [ ]:
!pip install --upgrade google-api-core>=2.10.2,<3.0.0dev
!pip install clipcap
!pip install diffusers
!pip install fairscale
!pip uninstall -y timm
!pip install timm==0.4.12
!pip install tensorboardX
!pip install torchscale
!pip install sacremoses
!pip install invisible_watermark transformers accelerate safetensors

In [2]:
import torch
from transformers import (GPT2Tokenizer, GPT2LMHeadModel)
import torch
import requests
from PIL import Image
import os
from torch import nn
import numpy as np
import torch.nn.functional as nnf
from typing import Tuple, List, Union, Optional, Callable
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from tqdm import tqdm, trange
import skimage.io as io
import PIL.Image
from IPython.display import Image 
from copy import deepcopy
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import torchvision.datasets as datasets
from torchvision.transforms.functional import InterpolationMode
import imageio
from skimage.transform import resize
#from diffusers import StableDiffusionPipeline
from transformers import pipeline, set_seed
#from kaggle_secrets import UserSecretsClient
import torchvision.transforms as transforms
import glob
import torch
import cv2
from PIL import Image
from tqdm import tqdm
import sys
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from types import MethodType
import torch.nn.functional as F
from transformers.image_utils import ChannelDimension

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [5]:
from diffusers import DiffusionPipeline

In [ ]:
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe = pipe.to(device)

In [ ]:
cap = datasets.CocoCaptions(root = '/kaggle/input/coco-image-caption/train2014/train2014',
                        annFile = '/kaggle/input/coco-image-caption/annotations_trainval2014/annotations/captions_train2014.json',
                        transform=transforms.Compose([
                        transforms.Resize(size=(255, 255))]))

In [8]:
def get_coco_dataset(id=3, for_attention=False):
  transform_totensor = transforms.ToTensor()
  img, target = cap[id]
  if for_attention:
    img = transform_totensor(img)
    return img, target
  else:
    return cap, target, img

def get_coco_dataset_for_sat(id, for_attention=False):
  pil_image, target = cap[id][0], cap[id][1]
  return cap, target, pil_image

In [9]:
def load_image(id, image_size, device, before=True, dataset='coco'):
  if dataset == 'coco':
    _, _, raw_image = get_coco_dataset(id)
  if dataset == 'flicker8k':
    _, _, raw_image = get_flickr8k_dataset(id)
  transform_pil_tensor = transforms.Compose([
        transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC),
        transforms.PILToTensor(),
    ])

  transform_tensor = transforms.Compose([
        transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))
    ])

  if before == True:
      image = transform_pil_tensor(raw_image).to(device)
  else:
      image = transform_tensor(raw_image).unsqueeze(0).to(device)
  return image

def load_image_for_sat(id, image_size, device, before=True, dataset='flicker8k'):
  if dataset == 'coco':
    folder_path = '/kaggle/input/coco-image-caption/train2014/train2014'
    files = os.listdir(folder_path)
    files = sorted(files)
    img = imageio.imread(folder_path + '/' + str(files[id]))
    '''
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')  
    plt.title("Selected Image")
    plt.show()
    '''
    if len(img.shape) == 2:
      img = img[:, :, np.newaxis] # add a new axis
      img = np.concatenate([img, img, img], axis=2)
    img = resize(img, (255, 255))
    img = img.transpose(2, 0, 1)
    if not before:
      # img = img/255
      img = torch.FloatTensor(img).to(device)
      #normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      #transform = transforms.Compose([normalize])
      #img = transform(img) # (3, image_size, image_size)
    else:
      img = torch.from_numpy(img)
  if dataset == 'flicker8k':
    captions_file = '/kaggle/input/flickr8k-token-txt/Flickr8k.token.txt'
    image_dir = '/kaggle/input/flickr8k/Images'
    with open(captions_file, 'r') as f:
      captions_data = f.readlines()

    captions_dict = {}
    for line in captions_data:
      parts = line.strip().split('\t')
      image_name = parts[0].split('#')[0]
      caption = parts[1]
      if image_name not in captions_dict:
        captions_dict[image_name] = []
      captions_dict[image_name].append(caption)
    image_ids = list(captions_dict.keys())
    if 0 <= id < len(image_ids):
      image_id = image_ids[id]
      captions = captions_dict[image_id]

      image_file = os.path.join(image_dir, image_id)
      image = Image.open(image_file)
      image_resized = image.resize((255, 255), Image.Resampling.LANCZOS)
      transform = transforms.Compose([transforms.ToTensor()])
      img = transform(image_resized)
    if not before:
      #img = img/255
      img = torch.FloatTensor(img).to(device)
      #normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      #transform = transforms.Compose([normalize])
      #img = transform(img) # (3, image_size, image_size)
  return img

In [10]:
class dataloader():
  def __init__(self, name):
    self.name = name
  def get_item(name, id, image_size):
    if name == 'coco':
      img = load_image(id, image_size, device, before=False)
      new_img = load_image(id, image_size, device)
      #print('image of dataloader: ', img)
      #print('new image of dataloader: ', new_img)
    if name == 'flicker8k':
      img = load_image(id, image_size, device, before=False, dataset=name)
      new_img = load_image_for_sat(id, image_size, device, dataset=name)
    return img, new_img
  def get_item_sat(name, id, image_size):
    if name == 'coco':
      img = load_image_for_sat(id, image_size, device, before=False, dataset=name)
      new_img = load_image_for_sat(id, image_size, device, dataset=name)
    if name == 'flicker8k':
      img = load_image_for_sat(id, image_size, device, before=False, dataset=name)
      new_img = load_image_for_sat(id, image_size, device, dataset=name)
    return img, new_img

In [ ]:
_, ori_image = dataloader.get_item_sat('coco', 1, image_size=224)
ori_image = ori_image.float()

In [ ]:
output_path_original = '/kaggle/working/original_images'
output_path_target = '/kaggle/working/target_images'

if not os.path.exists(output_path_original):
    os.makedirs(output_path_original)

if not os.path.exists(output_path_target):
    os.makedirs(output_path_target)

to_pil = transforms.ToPILImage()

for i in range(0, 1000):
    
    _, ori_image = dataloader.get_item_sat('coco', i, image_size=224)
    ori_image = ori_image.float()

    pil_ori_image = to_pil(ori_image.squeeze())

    caption = get_coco_dataset_for_sat(i+5)[1][0] 

    target_image = pipe(caption).images[0]  
    resized_target_image = target_image.resize((224, 224), Image.BICUBIC)

    safe_caption = caption.replace(' ', '_').replace('.', '').replace(',', '').replace('/', '_')

    pil_ori_image.save(os.path.join(output_path_original, f"{safe_caption}.png"))
    resized_target_image.save(os.path.join(output_path_target, f"{safe_caption}.png"))

<ipython-input-9-a777e93ee2a4>:28: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(folder_path + '/' + str(files[id]))


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]